# Swap The Knights

https://www.reddit.com/r/puzzles/comments/1xu6tbr/can_you_solve_this_in_how_many_moves/

A chess shortest path puzzle posted by u/Hit_learner.

Can be solved by a shortest path algorithm, specifically Dijkstra's algorithm.

- https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm

## Search Feasibility

A number of states is given by the number of ways we can distribute the knights (knights within each color are indistinguishable) on the 3-by-4 board.

In [ ]:
import math

In [ ]:
rows = 3
cols = 4
colors = 2
color_knights = 3

In [ ]:
nstates = math.prod(math.comb(rows*cols - c*color_knights, color_knights) for c in range(colors))
nstates

As we can see, the number of states is relatively small for a computer to search through (though large for a human to do by hand).

## Path Search

In fact our search space is small enough we can search every state to see if they are all reachable.  
Also we don't need to use something like A* to speed up the search, since we'll be visiting every node anyways.  

In [ ]:
import heapq
import itertools

In [ ]:
initial_state = tuple(tuple((k, (cols-1)*c) for k in range(color_knights)) for c in range(colors))
initial_state

In [ ]:
target_state = initial_state[::-1]
target_state

In [ ]:
must_alternate = True

In [ ]:
costs = {initial_state: 0}
count = {initial_state: 1}
prevs = {initial_state: None}
moves = {initial_state: None}
q = [(0, initial_state)]

while q:
    cost, curr = heapq.heappop(q)
    temp = [list(c) for c in curr]
    for c,k in itertools.product(range(colors), range(color_knights)):
        if must_alternate and cost % 2 != c:
            continue
        cr,cc = temp[c][k]
        for (lr,lc),(dr,dc) in itertools.product(
            itertools.permutations([1,2]), 
            itertools.product([-1,1], repeat=2)):
            nr, nc = cr + dr*lr, cc + dc*lc
            if any((nr, nc) in c for c in curr) or not ((0 <= nr < rows) and (0 <= nc < cols)):
                continue # overlap or outside
            temp[c][k] = (nr, nc)
            new_state = tuple(tuple(sorted(c)) for c in temp)
            if new_state not in prevs:
                moves[new_state] = (c, (cr,cc), (nr, nc))
                prevs[new_state] = curr
                costs[new_state] = cost+1
                count[new_state] = count[curr]
                heapq.heappush(q, (cost+1, new_state))
            elif costs[new_state] == cost+1:
                count[new_state] += count[curr]
            temp[c][k] = (cr, cc)
target_state in costs

## Solution Path

In [ ]:
color_map = ['white', 'black']
alpha_map = lambda c: chr(ord('a')+c)

curr = target_state
path = []
while curr in prevs:
    if moves[curr]:
        (c, (cr,cc), (nr, nc)) = moves[curr]
        path.append(color_map[c]+' '+alpha_map(cc)+str(3-cr)+'-'+alpha_map(nc)+str(3-nr))
    curr = prevs[curr]
path = path[::-1]
print('\n'.join(path))

len(path)

Reachable states:

In [ ]:
len(costs)

In [ ]:
nstates - len(costs)

In [ ]:
count[target_state]

In [ ]:
max(costs.values())

## Conclusion

- If alternating turns is not enforced:
    - All states are reachable
    - The optimal solution is 16 steps
- If alternating turns is enforced:
    - 28 states are not reachable
    - The optimal solution is 18 steps